#  분자 지문 (Molecular Fingerprints)

분자는 여러 가지 방식으로 표현할 수 있습니다. 이번 튜토리얼에서는 "분자 지문(molecular fingerprint)"이라고 부르는 표현 방식을 소개합니다. 이는 매우 단순한 표현이지만, 작은 약물 유사(drug-like) 분자에 대해서는 잘 동작하는 경우가 많습니다.

In [ ]:
!pip install -qq --pre deepchem

In [ ]:
# warning message 출력 안하기
import warnings
warnings.filterwarnings('ignore')

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

이제 `deepchem` 패키지를 import 하여 실습을 시작할 수 있습니다.


In [ ]:
import deepchem as dc
dc.__version__


# 지문(Fingerprint)이란 무엇인가?

딥러닝 모델은 거의 항상 숫자 배열을 입력으로 받습니다. 따라서 분자를 모델로 처리하려면, 각 분자를 어떤 식으로든 하나 이상의 숫자 배열로 표현해야 합니다.

(전부는 아니지만) 많은 종류의 모델은 입력 크기가 고정되어 있어야 합니다. 그런데 분자마다 원자(atom) 개수가 다르기 때문에, 이는 분자를 다룰 때 어려움이 됩니다. 이런 종류의 모델을 사용하려면, 크기가 제각각인 분자들을 고정된 크기의 배열로 표현해야 합니다.

지문(Fingerprint)은 바로 이러한 문제를 해결하기 위해 설계되었습니다. 지문은 고정된 길이의 배열이며, 각 원소(element)는 분자 안에 특정 특징(feature)이 존재하는지를 나타냅니다. 두 분자의 지문이 비슷하다면, 두 분자가 많은 공통 특징을 가지고 있다는 의미이고, 따라서 화학적 성질도 비슷할 가능성이 높습니다.

DeepChem은 "확장 연결성 지문(Extended Connectivity Fingerprint)", 줄여서 "ECFP"라고 부르는 특정 종류의 지문을 지원합니다. 이는 "원형 지문(circular fingerprint)"이라고 불리기도 합니다. ECFP 알고리즘은 먼저 각 원자를 그 원자가 가진 직접적인 속성과 결합(bond)만을 기준으로 분류하는 것에서 시작합니다. 이렇게 만들어진 고유한 패턴 하나하나가 곧 특징(feature)이 됩니다. 예를 들어 "수소 원자 2개와 무거운 원자(heavy atom) 2개에 결합된 탄소 원자"가 하나의 특징이 될 수 있으며, 이 특징을 포함하는 분자에 대해서는 지문의 해당 원소가 1로 설정됩니다. 이후 알고리즘은 점점 더 넓은 원형 이웃(circular neighborhood)을 살펴보면서 새로운 특징을 반복적으로 찾아냅니다. 특정 특징이 다른 두 특징과 결합되어 있으면 더 높은 수준의 특징이 되고, 이를 포함하는 분자에 대해 해당 원소가 설정됩니다. 이 과정은 정해진 횟수만큼 반복되며, 보통은 2회 반복합니다.

이제 ECFP로 특징화(featurize)된 데이터셋을 살펴보겠습니다.


In [ ]:
# Tox21 데이터셋을 ECFP 지문으로 특징화하여 불러옵니다.
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer='ECFP')
# 데이터셋은 학습용/검증용/테스트용으로 나뉘어 반환됩니다.
train_dataset, valid_dataset, test_dataset = datasets
print(train_dataset)


특징 배열 `X`의 형태(shape)는 (6264, 1024)입니다. 즉, 학습(training) 데이터셋에 6264개의 샘플이 있다는 뜻입니다. 각 샘플은 길이 1024의 지문으로 표현됩니다. 또한 레이블 배열 `y`의 형태가 (6264, 12)인 점에 주목하세요. 이것은 다중 작업(multitask) 데이터셋입니다. Tox21은 분자의 독성(toxicity)에 대한 정보를 담고 있습니다. 독성의 징후를 찾기 위해 12가지의 서로 다른 분석(assay)이 사용되었습니다. 데이터셋은 이 12가지 분석 결과를 각각 별개의 작업(task)으로 기록합니다.

이번에는 가중치(weight) 배열도 함께 살펴보겠습니다.


In [ ]:
# 가중치(weight) 배열을 살펴봅니다.
train_dataset.w


일부 원소가 0인 것을 확인할 수 있습니다. 이 가중치는 결측 데이터(missing data)를 표시하는 데 사용됩니다. 모든 분석이 실제로 모든 분자에 대해 수행된 것은 아닙니다. 어떤 샘플 또는 샘플/작업 쌍의 가중치를 0으로 설정하면, 학습(fitting)과 평가(evaluation) 과정에서 해당 항목이 무시됩니다. 즉, 손실 함수(loss function)나 다른 지표(metric)에 아무런 영향을 주지 않습니다.

나머지 대부분의 가중치는 1에 가깝지만 정확히 1은 아닙니다. 이는 각 작업에서 양성(positive) 샘플과 음성(negative) 샘플의 전체 가중치 균형을 맞추기 위한 것입니다. 모델을 학습할 때 우리는 12개 작업이 모두 동등하게 기여하기를 바라며, 각 작업 안에서도 양성 샘플과 음성 샘플에 동일한 가중치를 두고 싶습니다. 그렇지 않으면 모델이 "대부분의 학습 샘플은 독성이 없다"는 식으로만 학습해 버려, 다른 분자들도 독성이 없다고 판단하는 쪽으로 편향(bias)될 수 있습니다.

# 지문으로 모델 학습하기

이제 모델을 학습시켜 봅시다. 앞선 튜토리얼들에서는 `GraphConvModel`을 사용했는데, 이는 복잡한 입력 집합을 받는 비교적 복잡한 구조입니다. 반면 지문은 고정 길이 배열 하나로 매우 단순하기 때문에, 훨씬 더 간단한 종류의 모델을 사용할 수 있습니다.


In [ ]:
# 작업(task) 12개, 특징(feature) 1024개, 너비 1000짜리 은닉층 하나를 갖는 다중 작업 분류기를 만듭니다.
model = dc.models.MultitaskClassifier(n_tasks=12, n_features=1024, layer_sizes=[1000])


`MultitaskClassifier`는 완전 연결 층(fully connected layer)을 단순하게 쌓아 올린 모델입니다. 이번 예제에서는 너비(width)가 1000인 은닉층(hidden layer) 하나를 사용하도록 지정했습니다. 또한 각 입력이 1024개의 특징을 가지며, 12개의 서로 다른 작업에 대한 예측을 생성해야 한다고 알려주었습니다.

왜 각 작업마다 별도의 모델을 학습시키지 않을까요? 그렇게 할 수도 있지만, 여러 작업에 대해 하나의 모델을 함께 학습시키는 편이 더 잘 동작하는 경우가 많습니다. 이에 대한 예시는 이후 튜토리얼에서 살펴보겠습니다.

이제 모델을 학습시키고 평가해 봅시다.


In [ ]:
import numpy as np

# 모델을 10 에폭(epoch) 동안 학습시킵니다.
model.fit(train_dataset, nb_epoch=10)
# ROC-AUC 점수를 평가 지표로 사용합니다.
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
print('학습 데이터셋 점수:', model.evaluate(train_dataset, [metric], transformers))
print('테스트 데이터셋 점수:', model.evaluate(test_dataset, [metric], transformers))


이렇게 단순한 모델과 특징화 방식치고는 나쁘지 않은 성능입니다. 더 정교한 모델은 이 데이터셋에서 약간 더 나은 성능을 보이지만, 엄청나게 좋아지지는 않습니다.
